# Python Fundamentals & Data Types — Expert Interview Guide

Covers: Python data model, mutability, all built-in types, comprehensions, scope, and key gotchas.

> **Key mental model:** In Python, every name is a reference (pointer) to an object. Variables don't hold values — they point to objects in memory.

## 1. Object Identity: `id()`, `is` vs `==`, Interning

- `==` compares **values** (calls `__eq__`)

- `is` compares **identity** (same object in memory, same `id()`)

- Python interns small integers (-5 to 256) and short strings — they share the same object

In [ ]:
# id() returns memory address of object
a = 1000
b = 1000
print(f"a == b: {a == b}")   # True  (same value)
print(f"a is b: {a is b}")   # False (different objects, outside intern range)
print(f"id(a)={id(a)}, id(b)={id(b)}")

# Small integer interning (-5 to 256)
x = 100
y = 100
print(f"\n100: x is y = {x is y}")  # True — interned

x2 = 1000
y2 = 1000
print(f"1000: x is y = {x2 is y2}")  # False — not interned (CPython impl detail)

# String interning
s1 = "hello"
s2 = "hello"
print(f"\n'hello': s1 is s2 = {s1 is s2}")  # True — interned (compile-time constant)

s3 = "hello world"
s4 = "hello world"
print(f"'hello world': s3 is s4 = {s3 is s4}")  # Usually True for literals

# None, True, False are singletons
print(f"\nNone is None: {None is None}")    # Always True
print(f"True is True: {True is True}")       # Always True

# The right way to check None
val = None
print(f"val is None: {val is None}")         # Correct
print(f"val == None: {val == None}")         # Works but bad style

> **Interview Insight:** Never use `is` to compare values — only use it for `None`, `True`, `False`. Integer interning is a CPython implementation detail, not guaranteed by the Python spec.

## 2. Mutable vs Immutable Types

| Immutable | Mutable |

|---|---|

| `int`, `float`, `complex`, `bool` | `list`, `dict`, `set` |

| `str`, `bytes`, `tuple`, `frozenset` | `bytearray` |

**Immutable objects** cannot be changed after creation — any 'modification' creates a new object.  

**Mutable objects** can be changed in-place — all references see the change.

In [ ]:
# Immutable: 'changing' creates a new object
s = "hello"
print(f"Before: id={id(s)}")
s += " world"
print(f"After:  id={id(s)}")  # Different id!

# Mutable: changes in-place
lst = [1, 2, 3]
print(f"\nList before: id={id(lst)}")
lst.append(4)
print(f"List after:  id={id(lst)}")  # Same id!

# The mutable default argument gotcha
def append_to(item, target=[]):   # BAD: mutable default shared across calls
    target.append(item)
    return target

print(f"\nCall 1: {append_to(1)}")  # [1]
print(f"Call 2: {append_to(2)}")    # [1, 2]  <- bug!
print(f"Call 3: {append_to(3)}")    # [1, 2, 3] <- bug!

def append_to_fixed(item, target=None):  # GOOD
    if target is None:
        target = []
    target.append(item)
    return target

print(f"\nFixed call 1: {append_to_fixed(1)}")  # [1]
print(f"Fixed call 2: {append_to_fixed(2)}")    # [2]

# Tuple of mutables — tuple is immutable, but contents can change
t = ([1, 2], [3, 4])
t[0].append(99)  # list inside tuple is still mutable!
print(f"\nTuple with mutable: {t}")  # ([1, 2, 99], [3, 4])

> **Interview Insight:** The mutable default argument bug is one of the most common Python gotchas in interviews. Always use `None` as default for mutable arguments.

## 3. All Built-in Types Overview

In [ ]:
None       — singleton null value
bool       — True/False (subclass of int)
int        — arbitrary precision integer
float      — 64-bit IEEE 754 double
complex    — complex(real, imag)
str        — immutable Unicode text
bytes      — immutable byte sequence
bytearray  — mutable byte sequence
list       — mutable ordered sequence
tuple      — immutable ordered sequence
range      — lazy integer sequence
dict       — hash map (ordered since 3.7)
set        — mutable unordered unique collection
frozenset  — immutable set

In [ ]:
# Numeric types
print("=== Numeric Types ===")
print(type(42), type(3.14), type(1+2j), type(True))
print(f"bool is int subclass: {issubclass(bool, int)}")
print(f"True + True = {True + True}")  # 2 — bool arithmetic!

# Arbitrary precision int
big = 2 ** 100
print(f"\n2^100 = {big}")

# Float precision
print(f"\n0.1 + 0.2 = {0.1 + 0.2}")        # 0.30000000000000004
print(f"0.1 + 0.2 == 0.3: {0.1 + 0.2 == 0.3}")  # False!

from decimal import Decimal
print(f"Decimal: {Decimal('0.1') + Decimal('0.2')}")  # 0.3 exact

from fractions import Fraction
print(f"Fraction: {Fraction(1,3) + Fraction(1,6)}")  # 1/2

# Bytes vs str
print("\n=== Bytes vs Str ===")
s = "hello"
b = b"hello"
ba = bytearray(b"hello")
print(type(s), type(b), type(ba))
print(f"str encode: {s.encode('utf-8')}")
print(f"bytes decode: {b.decode('utf-8')}")
ba[0] = 72  # H
print(f"bytearray modified: {ba}")

# Range — lazy
r = range(1_000_000)
print(f"\nrange size in memory: {__import__('sys').getsizeof(r)} bytes")
print(f"list size: {__import__('sys').getsizeof(list(range(1000)))} bytes")

> **Interview Insight:** `bool` is a subclass of `int` — `True == 1` and `False == 0`. This means `True + True == 2` and `sum([True, False, True]) == 2`. Used for counting booleans in a list.

## 4. String Methods & f-strings

In [ ]:
# Key string methods
s = "  Hello, World!  "
print(s.strip())           # "Hello, World!"
print(s.lower())           # "  hello, world!  "
print(s.upper())           # "  HELLO, WORLD!  "
print(s.strip().replace(",", ""))
print("hello world".title())        # "Hello World"
print("hello world".split())        # ["hello", "world"]
print(",".join(["a","b","c"]))       # "a,b,c"
print("hello".startswith("hel"))    # True
print("hello".find("ll"))           # 2
print("hello".count("l"))           # 2
print("  ".isspace())               # True
print("abc123".isalnum())           # True

# f-strings (Python 3.6+)
name, price, pi = "widget", 9.99, 3.14159
print(f"\nItem: {name!r}")                    # repr
print(f"Price: ${price:.2f}")                  # 2 decimal places
print(f"Pi: {pi:.4f}")                         # 4 decimals
print(f"Binary: {255:08b}")                    # 11111111
print(f"Hex: {255:#x}")                        # 0xff
print(f"Thousands: {1_000_000:,}")             # 1,000,000
print(f"Percent: {0.754:.1%}")                 # 75.4%
print(f"Width: {name:>10}")                    # right-align
print(f"Width: {name:<10}|")                   # left-align

# f-string self-documenting (Python 3.8+)
x = 42
print(f"{x=}")  # x=42

# String interning
import sys
s1 = sys.intern("my_long_identifier")
s2 = sys.intern("my_long_identifier")
print(f"\nIntern: s1 is s2 = {s1 is s2}")  # True

> **Interview Insight:** f-strings are the fastest string formatting method in Python 3. `str.format()` is ~2x slower, `%` formatting is deprecated style. Use `f'{x=}'` for debugging — prints both the expression and value.

## 5. List, Tuple, Dict, Set — Core Operations

In [ ]:
from collections import defaultdict, OrderedDict, Counter, ChainMap

# List
lst = [3, 1, 4, 1, 5, 9, 2, 6]
print(sorted(lst))                  # new sorted list
lst.sort(reverse=True)              # in-place sort
print(lst)
print(lst[1:4])                     # slice [1, 4, 5]
print(lst[::2])                     # every 2nd element

# Dict — ordered since Python 3.7
d = {"a": 1, "b": 2, "c": 3}
print(d.get("x", 0))               # 0 (safe get with default)
print(d.keys(), d.values(), d.items())
d2 = {"c": 30, "d": 4}
merged = d | d2                     # merge (Python 3.9+)
print(merged)

# defaultdict
dd = defaultdict(list)
dd["fruits"].append("apple")
dd["fruits"].append("banana")
print(dd)

# Counter
words = ["apple", "banana", "apple", "cherry", "banana", "apple"]
c = Counter(words)
print(c.most_common(2))            # [('apple', 3), ('banana', 2)]
print(c["apple"])                  # 3

# ChainMap — search multiple dicts as one
defaults = {"color": "red", "size": "M"}
user_prefs = {"color": "blue"}
cm = ChainMap(user_prefs, defaults)
print(cm["color"])  # blue (user wins)
print(cm["size"])   # M (from defaults)

# Set operations
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}
print(a | b)   # union
print(a & b)   # intersection
print(a - b)   # difference
print(a ^ b)   # symmetric difference (in one but not both)

# frozenset — hashable, usable as dict key
fs = frozenset([1, 2, 3])
d = {fs: "value"}
print(d[frozenset([1, 2, 3])])

> **Interview Insight:** `dict` maintains insertion order since Python 3.7 (CPython 3.6). `Counter` is a subclass of `dict`. `ChainMap` is perfect for layered config (CLI args > env vars > defaults) without copying.

## 6. Comprehensions vs Generator Expressions

In [ ]:
import sys

# List comprehension — eager, stores all results
squares = [x**2 for x in range(10)]
print(squares)

# Dict comprehension
word_lengths = {word: len(word) for word in ["apple", "banana", "cherry"]}
print(word_lengths)

# Set comprehension — unique values
unique_lengths = {len(word) for word in ["apple", "banana", "cherry", "fig"]}
print(unique_lengths)

# Nested comprehension (matrix transpose)
matrix = [[1,2,3],[4,5,6],[7,8,9]]
transposed = [[row[i] for row in matrix] for i in range(3)]
print(transposed)

# Conditional comprehension
evens = [x for x in range(20) if x % 2 == 0]
print(evens)

# Generator expression — lazy, one element at a time
gen = (x**2 for x in range(10))
print(type(gen))
print(sys.getsizeof(squares))  # stores all 10 values
print(sys.getsizeof(gen))      # tiny — just the generator object

# Use generators for large datasets
def read_large_file(n=1_000_000):
    return (x**2 for x in range(n))  # lazy

gen = read_large_file()
print(next(gen))   # 0
print(next(gen))   # 1
print(sum(x for x in range(100) if x % 2 == 0))  # sum of even squares

# Walrus operator := (Python 3.8+)
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
filtered = [y for x in data if (y := x**2) > 25]
print(filtered)  # [36, 49, 64, 81, 100]

> **Interview Insight:** Use list comprehensions when you need random access or length. Use generators when iterating once over large data — they use O(1) memory vs O(n). Never use a list comprehension if you only iterate once.

## 7. Sequence Unpacking & Extended Unpacking

In [ ]:
# Basic unpacking
a, b, c = 1, 2, 3
print(a, b, c)

# Swap without temp variable
a, b = b, a
print(a, b)  # 2, 1

# Extended unpacking with *
first, *rest = [1, 2, 3, 4, 5]
print(first, rest)  # 1, [2, 3, 4, 5]

*init, last = [1, 2, 3, 4, 5]
print(init, last)  # [1, 2, 3, 4], 5

first, *middle, last = [1, 2, 3, 4, 5]
print(first, middle, last)  # 1, [2, 3, 4], 5

# Nested unpacking
(a, b), c = (1, 2), 3
print(a, b, c)

# Unpacking in for loops
pairs = [(1, "a"), (2, "b"), (3, "c")]
for num, letter in pairs:
    print(f"{num}: {letter}")

# Dict unpacking
d1 = {"a": 1, "b": 2}
d2 = {"c": 3, "d": 4}
merged = {**d1, **d2}
print(merged)

def func(a, b, c): return a + b + c
args = [1, 2, 3]
kwargs = {"a": 1, "b": 2, "c": 3}
print(func(*args))
print(func(**kwargs))

> **Interview Insight:** `first, *rest = iterable` works on any iterable, not just lists. The `*` variable always collects into a `list`. This is the Pythonic way to split head/tail of sequences.

## 8. LEGB Scope Rule

**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

Python searches for names in this order. `global` and `nonlocal` keywords modify this.

In [ ]:
x = "global"

def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print(f"inner sees: {x}")  # local

    def inner_no_local():
        print(f"no_local sees: {x}")  # enclosing (not local)

    def inner_nonlocal():
        nonlocal x  # modify enclosing scope
        x = "modified by inner"

    inner()
    inner_no_local()
    print(f"Before nonlocal: {x}")
    inner_nonlocal()
    print(f"After nonlocal:  {x}")

outer()
print(f"Global unchanged: {x}")

# global keyword
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(f"\nCounter: {counter}")  # 2

# UnboundLocalError gotcha
def broken():
    print(x)  # tries to print 'x' but Python sees 'x = ...' below
    x = 10    # this makes x a local variable
# broken()  # UnboundLocalError!

# Built-in scope
print(f"\nBuilt-in len: {len([1,2,3])}")
len = "shadowing built-in!"  # BAD
print(f"Shadowed len: {len}")
del len  # restore
print(f"Restored len: {len([1,2,3])}")

> **Interview Insight:** The `UnboundLocalError` happens because Python determines scope at **compile time** — if any assignment to `x` exists in a function, `x` is local throughout the function, including lines before the assignment.

## 9. Truthiness, Falsy Values & Short-Circuit Evaluation

In [ ]:
# Falsy values in Python
falsy_values = [False, None, 0, 0.0, 0j, "", b"", [], (), {}, set(), range(0)]
for val in falsy_values:
    print(f"bool({val!r:15}) = {bool(val)}")

# Everything else is truthy
print(f"\nbool([0]) = {bool([0])}")    # True — non-empty list
print(f"bool('False') = {bool('False')}")  # True — non-empty string

# Short-circuit evaluation
def expensive():
    print("  expensive() called!")
    return True

# 'and' stops at first falsy
result = False and expensive()    # expensive() NOT called
print(f"\nFalse and ...: {result}")

result = True and expensive()     # expensive() called
print(f"True and ...: {result}")

# 'or' stops at first truthy
result = True or expensive()      # expensive() NOT called
print(f"\nTrue or ...: {result}")

result = False or expensive()     # expensive() called
print(f"False or ...: {result}")

# Idiomatic patterns using truthiness
name = ""
display = name or "Anonymous"     # "Anonymous"
print(f"\nName: {display}")

data = None
count = data and len(data)        # None (avoids TypeError)
print(f"Count: {count}")

# __bool__ and __len__ for custom truthiness
class SmartList:
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __bool__(self):
        return len(self.items) > 0

sl = SmartList([])
print(f"\nEmpty SmartList is truthy: {bool(sl)}")  # False

> **Interview Insight:** Short-circuit returns the **actual value** (not just True/False). `x = a or b` sets `x` to `a` if `a` is truthy, else `b`. This is the Pythonic null-coalescing pattern.